In [ ]:
import io
from pathlib import Path
import re
import zipfile

import geopandas as gpd
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

In [ ]:
# folder with simplified (zip) and original (geojson) shapes
shapes_folder = "./data_in/geo_sight/cod_shapes/"

# thresholds to flag geometries differing significantly
iou_threshold = 0.90  # min intersection over union
area_ratio_tol = 0.10  # max relative area difference (simplified vs original)

# min overlap (fraction of the smaller unit) to link units across files
# NOTE: drops border slivers due to simplification
edge_min_frac = 0.05

# equal area CRS to compute areas
equal_area_crs = "EPSG:6933"

In [ ]:
def find_original(folder, iso3, level):
    """Original (non zip) geojson with iso3 + "_" and adm level in name.

    Case insensitive; assumes one or none. Optional "in" and "_" in
    "adm" + level, e.g. adm2, admin2, adm_2 (as in template_pcodes).
    """
    pattern = re.compile(rf"adm(in)?_?{level}(?!\d)", re.IGNORECASE)
    matches = [
        f for f in sorted(Path(folder).iterdir())
        if f.is_file()
        and f.suffix.lower() == ".geojson"
        and f"{iso3}_".lower() in f.name.lower()
        and pattern.search(f.name)
    ]
    return matches[0] if matches else None


def read_simplified(zip_path):
    """GeoDataFrame from the geojson inside the simplified zip file."""
    with zipfile.ZipFile(zip_path, "r") as zip_file:
        member = [
            n for n in zip_file.namelist()
            if n.lower().endswith(".geojson")
        ][0]
        return gpd.read_file(io.BytesIO(zip_file.read(member)))


def get_col(gdf, name):
    """Column in gdf matching name case insensitive (None if missing)."""
    found = [c for c in gdf.columns if c.lower() == name.lower()]
    return found[0] if found else None


def prepare(gdf, pcode_col):
    """Valid geometries dissolved by pcode, in equal area CRS."""
    gdf = gdf[[pcode_col, "geometry"]].copy()
    gdf[pcode_col] = gdf[pcode_col].astype(str).str.strip()
    gdf["geometry"] = gdf.make_valid()
    return gdf.dissolve(by=pcode_col).to_crs(equal_area_crs)


def overlap_table(gdf_s, gdf_o, pcode_s, pcode_o):
    """Overlaps between units of both files (equal area CRS).

    Keeps pairs where the intersection is at least edge_min_frac of the
    smaller unit. Returns edges and the pcodes of each file.
    """
    geo_s = prepare(gdf_s, pcode_s).geometry
    geo_o = prepare(gdf_o, pcode_o).geometry

    idx_s, idx_o = geo_o.sindex.query(geo_s.values, predicate="intersects")
    inter = geo_s.iloc[idx_s].intersection(
        geo_o.iloc[idx_o], align=False
    ).area.to_numpy()
    area_s = geo_s.area.to_numpy()[idx_s]
    area_o = geo_o.area.to_numpy()[idx_o]

    edges = pd.DataFrame({
        "pcode_s": geo_s.index[idx_s],
        "pcode_o": geo_o.index[idx_o],
        "iou": inter / (area_s + area_o - inter),
        "frac_s": inter / area_s,
        "frac_o": inter / area_o,
        "area_ratio": area_s / area_o,
    })
    edges = edges[
        edges[["frac_s", "frac_o"]].max(axis=1) >= edge_min_frac
    ].reset_index(drop=True)
    return edges, list(geo_s.index), list(geo_o.index)


def classify_relations(edges, pcodes_s, pcodes_o):
    """Crosswalk of units across files from connected overlaps.

    relation: same_pcode, pcode_changed, pcode_interchanged (1 to 1);
    split (1 simplified to n original); merge (n to 1); complex (n to m);
    no_counterpart_s / no_counterpart_o (no overlap in the other file).
    """
    node_s = {p: i for i, p in enumerate(pcodes_s)}
    node_o = {p: len(pcodes_s) + i for i, p in enumerate(pcodes_o)}
    n_nodes = len(pcodes_s) + len(pcodes_o)
    rows = edges["pcode_s"].map(node_s).to_numpy()
    cols = edges["pcode_o"].map(node_o).to_numpy()
    _, labels = connected_components(
        coo_matrix((np.ones(len(edges)), (rows, cols)), shape=(n_nodes, n_nodes)),
        directed=False,
    )

    cw = edges.assign(component=labels[rows])
    n_s = cw.groupby("component")["pcode_s"].transform("nunique")
    n_o = cw.groupby("component")["pcode_o"].transform("nunique")
    cw["relation"] = np.select(
        [(n_s == 1) & (n_o == 1), n_s == 1, n_o == 1],
        ["one_to_one", "split", "merge"],
        default="complex",
    )

    # one to one: same pcode, changed, or interchanged (pcodes reused)
    one = cw["relation"] == "one_to_one"
    changed = one & (cw["pcode_s"] != cw["pcode_o"])
    cw.loc[one, "relation"] = "same_pcode"
    cw.loc[changed, "relation"] = "pcode_changed"
    interchanged = (
        changed
        & cw["pcode_s"].isin(cw.loc[changed, "pcode_o"])
        & cw["pcode_o"].isin(cw.loc[changed, "pcode_s"])
    )
    cw.loc[interchanged, "relation"] = "pcode_interchanged"

    # units without overlap in the other file
    lone_s = sorted(set(pcodes_s) - set(cw["pcode_s"]))
    lone_o = sorted(set(pcodes_o) - set(cw["pcode_o"]))
    lone = pd.concat([
        pd.DataFrame({"pcode_s": lone_s, "relation": "no_counterpart_s"}),
        pd.DataFrame({"pcode_o": lone_o, "relation": "no_counterpart_o"}),
    ], ignore_index=True)
    lone["component"] = labels.max() + 1 + np.arange(len(lone))

    return pd.concat([cw, lone], ignore_index=True).sort_values(
        ["relation", "component"]
    ).reset_index(drop=True)


one_to_one_rel = ["same_pcode", "pcode_changed", "pcode_interchanged"]


def print_reconciliation(cw, max_rows=20):
    """Proposal to reconcile pcodes across files from geometry matches."""
    if set(cw["relation"]) <= {"same_pcode"}:
        print("Pcodes consistent across files, nothing to reconcile")
        return

    cases = [
        ("pcode_changed", "Same shape, different pcode: recode original "
         "pcode (pcode_o) to simplified pcode (pcode_s)",
         ["pcode_o", "pcode_s", "iou"]),
        ("pcode_interchanged", "Same shapes, pcodes interchanged between "
         "units: recode one file with this mapping",
         ["pcode_o", "pcode_s", "iou"]),
        ("split", "Split: a simplified unit is divided in the original; "
         "aggregate pcode_o to pcode_s (frac_o ~ share of pcode_o in pcode_s)",
         ["component", "pcode_s", "pcode_o", "frac_s", "frac_o"]),
        ("merge", "Merge: several simplified units are one in the original; "
         "aggregate pcode_s to pcode_o (frac_s ~ share of pcode_s in pcode_o)",
         ["component", "pcode_s", "pcode_o", "frac_s", "frac_o"]),
        ("complex", "Different subdivision (n to m): no clean crosswalk, "
         "manual review (area fractions as allocation weights)",
         ["component", "pcode_s", "pcode_o", "frac_s", "frac_o"]),
        ("no_counterpart_s", "Units only in simplified (no overlap in original)",
         ["pcode_s"]),
        ("no_counterpart_o", "Units only in original (no overlap in simplified)",
         ["pcode_o"]),
    ]
    for relation, text, cols in cases:
        sub = cw[cw["relation"] == relation]
        if sub.empty:
            continue
        print(f"{text} [{len(sub)} rows]")
        print(sub[cols].round(3).head(max_rows).to_string(index=False))
        if len(sub) > max_rows:
            print(f"... {len(sub) - max_rows} more rows")

In [ ]:
summary_rows = []
crosswalks = {}  # crosswalk per file, to inspect / export

for zip_path in sorted(Path(shapes_folder).glob("geojson_*_adm*_simpl*.zip")):
    print("=" * 70)
    print(zip_path.name)

    parsed = re.match(
        r"geojson_([A-Za-z]{3})_.*?adm(\d+)_simpl", zip_path.name
    )
    if parsed is None:
        print("Note file name not parsed (iso3 / adm level), skipping")
        continue
    iso3, level = parsed.groups()

    # equivalent original geojson (same iso3 and adm level)
    original = find_original(shapes_folder, iso3, level)
    if original is None:
        print("No corresponding original geojson, skipping analysis")
        continue
    print(f"Original: {original.name}")

    gdf_s = read_simplified(zip_path)
    gdf_o = gpd.read_file(original)

    # pcode column (case insensitive); same name expected in both files
    pcode_name = f"adm{level}_pcode"
    pcode_s = get_col(gdf_s, pcode_name)
    pcode_o = get_col(gdf_o, pcode_name)
    if pcode_s is None or pcode_o is None:
        print(f"Note column '{pcode_name}' not found (case insensitive) in:")
        if pcode_s is None:
            print(f"  simplified, columns: {list(gdf_s.columns)}")
        if pcode_o is None:
            print(f"  original, columns: {list(gdf_o.columns)}")
        continue

    # pcodes comparison
    set_s = set(gdf_s[pcode_s].astype(str).str.strip())
    set_o = set(gdf_o[pcode_o].astype(str).str.strip())
    pcodes_equal = set_s == set_o
    print(f"Pcodes: {len(set_s)} simplified, {len(set_o)} original")
    if pcodes_equal:
        print("Pcodes are the same in both files")
    else:
        only_s = sorted(set_s - set_o)
        only_o = sorted(set_o - set_s)
        print(f"Pcodes only in simplified ({len(only_s)}): {only_s[:20]}")
        print(f"Pcodes only in original ({len(only_o)}): {only_o[:20]}")

    # match shapes across files (regardless of pcodes)
    edges, all_s, all_o = overlap_table(gdf_s, gdf_o, pcode_s, pcode_o)
    cw = classify_relations(edges, all_s, all_o)
    crosswalks[zip_path.name] = cw

    # components by relation (rows for no counterparts)
    rel_counts = cw.groupby("relation")["component"].nunique()
    print("Matches by relation (n components):")
    print(rel_counts.to_string())

    # shape similarity for 1 to 1 matches (same shape, regardless of pcode)
    res = cw[cw["relation"].isin(one_to_one_rel)]
    flagged = res[
        (res["iou"] < iou_threshold)
        | ((res["area_ratio"] - 1).abs() > area_ratio_tol)
    ]
    if len(res) > 0:
        print(
            f"1 to 1 IoU: median {res['iou'].median():.4f}, "
            f"min {res['iou'].min():.4f}; "
            f"area ratio: min {res['area_ratio'].min():.4f}, "
            f"max {res['area_ratio'].max():.4f}"
        )
        print(f"Flagged (IoU < {iou_threshold} or area diff > "
              f"{area_ratio_tol:.0%}): {len(flagged)}")
        if len(flagged) > 0:
            print(flagged.sort_values("iou")[
                ["pcode_s", "pcode_o", "iou", "area_ratio"]
            ].head(5).round(4))

    print_reconciliation(cw)

    summary_rows.append({
        "simplified": zip_path.name,
        "original": original.name,
        "n_simplified": len(set_s),
        "n_original": len(set_o),
        "pcodes_equal": pcodes_equal,
        **{r: rel_counts.get(r, 0) for r in [
            "same_pcode", "pcode_changed", "pcode_interchanged", "split",
            "merge", "complex", "no_counterpart_s", "no_counterpart_o",
        ]},
        "median_iou": res["iou"].median() if len(res) else np.nan,
        "min_iou": res["iou"].min() if len(res) else np.nan,
        "n_flagged": len(flagged),
    })

In [ ]:
# summary of all analysed files
# NOTE: crosswalks[<zip name>] has the unit matches per file
pd.DataFrame(summary_rows).round(4)